In [ ]:
# ── 2. CLEAN HOSPITAL GENERAL INFO ───────────────────────────────────────────

hosp_clean = hospital_info[[
    'Facility ID',
    'Facility Name',
    'State',
    'Hospital Type',
    'Hospital Ownership',
    'Hospital overall rating'
]].copy()

hosp_clean.columns = [
    'facility_id',
    'facility_name',
    'state',
    'hospital_type',
    'hospital_ownership',
    'overall_rating'
]

# convert rating to numeric - 'Not Available' becomes NaN
hosp_clean['overall_rating'] = pd.to_numeric(
    hosp_clean['overall_rating'], errors='coerce'
)

# standardize facility ID to 6-digit string with leading zeros
# e.g. 10001 becomes 010001
hosp_clean['facility_id'] = hosp_clean['facility_id'].astype(str).str.zfill(6)

print("\nHospital general info cleaned:")
print(hosp_clean.dtypes)
print("Missing ratings:", hosp_clean['overall_rating'].isnull().sum())


# ── 3. CLEAN READMISSION PROGRAM FILE ────────────────────────────────────────

# convert numeric columns - 'Too Few to Report' becomes NaN
readm_by_hospital['Excess Readmission Ratio'] = pd.to_numeric(
    readm_by_hospital['Excess Readmission Ratio'], errors='coerce'
)
readm_by_hospital['Predicted Readmission Rate'] = pd.to_numeric(
    readm_by_hospital['Predicted Readmission Rate'], errors='coerce'
)
readm_by_hospital['Expected Readmission Rate'] = pd.to_numeric(
    readm_by_hospital['Expected Readmission Rate'], errors='coerce'
)
readm_by_hospital['Number of Discharges'] = pd.to_numeric(
    readm_by_hospital['Number of Discharges'], errors='coerce'
)

# standardize facility ID to match hosp_clean
readm_by_hospital['Facility ID'] = readm_by_hospital['Facility ID'].astype(str).str.zfill(6)


# ── 4. COLLAPSE READMISSION FILE TO ONE ROW PER HOSPITAL ─────────────────────

# all six conditions - general hospital quality summary
all_conditions_summary = readm_by_hospital.groupby(['Facility ID', 'State']).agg(
    avg_excess_readm_ratio   = ('Excess Readmission Ratio',   'mean'),
    avg_predicted_readm_rate = ('Predicted Readmission Rate', 'mean'),
    avg_expected_readm_rate  = ('Expected Readmission Rate',  'mean'),
    num_conditions_measured  = ('Measure Name',               'count')
).reset_index().round(4)

# heart failure and COPD only - diabetes-relevant proxy
# diabetic patients have significantly higher comorbidity rates for both
diabetes_relevant = readm_by_hospital[
    readm_by_hospital['Measure Name'].isin(['READM-30-HF-HRRP', 'READM-30-COPD-HRRP'])
]

diabetic_proxy_summary = diabetes_relevant.groupby(['Facility ID', 'State']).agg(
    avg_excess_readm_hf_copd    = ('Excess Readmission Ratio',   'mean'),
    avg_predicted_readm_hf_copd = ('Predicted Readmission Rate', 'mean'),
    avg_expected_readm_hf_copd  = ('Expected Readmission Rate',  'mean')
).reset_index().round(4)

print("\nAll conditions summary shape:    ", all_conditions_summary.shape)
print("Diabetic proxy summary shape:    ", diabetic_proxy_summary.shape)


# ── 5. JOIN ALL THREE TABLES INTO ONE ENRICHED HOSPITAL TABLE ────────────────

# first join: hospital general info + all conditions summary
hosp_enriched = hosp_clean.merge(
    all_conditions_summary,
    left_on='facility_id',
    right_on='Facility ID',
    how='left'
).drop(columns=['Facility ID', 'State'])

hosp_enriched = hosp_enriched.rename(columns={'State_x': 'state'})

# second join: add the diabetes-relevant proxy columns
hosp_enriched = hosp_enriched.merge(
    diabetic_proxy_summary[['Facility ID',
                             'avg_excess_readm_hf_copd',
                             'avg_predicted_readm_hf_copd',
                             'avg_expected_readm_hf_copd']],
    left_on='facility_id',
    right_on='Facility ID',
    how='left'
).drop(columns=['Facility ID'])

print("\nFinal enriched hospital table shape:", hosp_enriched.shape)
print(hosp_enriched.head(3))


# ── 6. COMPUTE STATE-LEVEL AVERAGES ──────────────────────────────────────────

state_quality = hosp_enriched.groupby('state').agg(
    num_hospitals            = ('facility_id',              'count'),
    avg_star_rating          = ('overall_rating',           'mean'),
    avg_excess_readm_ratio   = ('avg_excess_readm_ratio',   'mean'),
    avg_excess_readm_hf_copd = ('avg_excess_readm_hf_copd', 'mean')
).reset_index().round(2)

print("\nState quality table shape:", state_quality.shape)
print(state_quality.sort_values('avg_star_rating', ascending=False).head(5))


# ── 7. COMPUTE NATIONAL BENCHMARKS ───────────────────────────────────────────

national_benchmark = pd.DataFrame([{
    'total_hospitals_rated':         int(hosp_enriched['overall_rating'].count()),
    'national_avg_star_rating':      round(hosp_enriched['overall_rating'].mean(), 2),
    'national_avg_excess_readm':     round(hosp_enriched['avg_excess_readm_ratio'].mean(), 4),
    'national_avg_predicted_readm':  round(hosp_enriched['avg_predicted_readm_rate'].mean(), 2),
    'national_avg_excess_hf_copd':   round(hosp_enriched['avg_excess_readm_hf_copd'].mean(), 4),
    'pct_hospitals_below_expected':  round(
        (hosp_enriched['avg_excess_readm_ratio'] < 1.0).sum() /
         hosp_enriched['avg_excess_readm_ratio'].count() * 100, 1
    )
}])

print("\nNational Benchmarks:")
print(national_benchmark.T)


# ── 8. SAVE ALL TABLES TO SQLITE ─────────────────────────────────────────────

os.makedirs('data', exist_ok=True)
connection = sqlite3.connect('data/diabetic_data.db')

df.to_sql(
    'diabetic_data', connection, if_exists='replace', index=False
)
hosp_enriched.to_sql(
    'hospital_quality', connection, if_exists='replace', index=False
)
state_quality.to_sql(
    'state_quality', connection, if_exists='replace', index=False
)
national_benchmark.to_sql(
    'national_benchmark', connection, if_exists='replace', index=False
)

connection.close()
print("\nAll four tables saved to data/diabetic_data.db")


# ── 9. VERIFY EVERYTHING LANDED CORRECTLY ────────────────────────────────────

connection = sqlite3.connect('data/diabetic_data.db')

audit = pd.read_sql("""
    SELECT 'diabetic_data'      AS table_name, COUNT(*) AS rows FROM diabetic_data
    UNION ALL
    SELECT 'hospital_quality'   AS table_name, COUNT(*) AS rows FROM hospital_quality
    UNION ALL
    SELECT 'state_quality'      AS table_name, COUNT(*) AS rows FROM state_quality
    UNION ALL
    SELECT 'national_benchmark' AS table_name, COUNT(*) AS rows FROM national_benchmark
""", connection)

connection.close()
print("\nDatabase audit:")
print(audit)

In [ ]:
import pandas as pd
import sqlite3

# your UCI patient data - already loaded as df from previous days
# confirming it's still in memory
print("UCI data shape:", df.shape)

# load the CMS hospital general information
hospital_info = pd.read_csv(
    r'data\hospitals_current_data\Hospital_General_Information.csv')

# load the CMS readmission reduction program data
# this file has hospital-level readmission penalty information
readm_by_hospital = pd.read_csv(
    r'data\hospitals_current_data\FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv'
)

print("Hospital general info shape:", hospital_info.shape)
print("Readmission program shape:", readm_by_hospital.shape)

# readmission program shape is large due to the fact that 5 readmission types are measured for each hospital

In [ ]:
# we only want columns that are meaningful for enrichment
# dropping footnote columns, address details, phone numbers etc.
hosp_clean = hospital_info[[
    'Facility ID',
    'Facility Name', 
    'State',
    'Hospital Type',
    'Hospital Ownership',
    'Hospital overall rating'
]].copy()

# rename to lowercase with underscores - consistent with your UCI column style
hosp_clean.columns = [
    'facility_id',
    'facility_name',
    'state',
    'hospital_type',
    'hospital_ownership',
    'overall_rating'
]

# 'Hospital overall rating' contains strings like 'Not Available'
# pd.to_numeric with errors='coerce' turns those into NaN instead of crashing
hosp_clean['overall_rating'] = pd.to_numeric(
    hosp_clean['overall_rating'], errors='coerce'
)

print(hosp_clean.dtypes)
print("\nMissing values:\n", hosp_clean.isnull().sum())

In [ ]:
# ── 2. CLEAN HOSPITAL GENERAL INFO ───────────────────────────────────────────

hosp_clean = hospital_info[[
    'Facility ID',
    'Facility Name',
    'State',
    'Hospital Type',
    'Hospital Ownership',
    'Hospital overall rating'
]].copy()

hosp_clean.columns = [
    'facility_id',
    'facility_name',
    'state',
    'hospital_type',
    'hospital_ownership',
    'overall_rating'
]

# convert rating to numeric - 'Not Available' becomes NaN
hosp_clean['overall_rating'] = pd.to_numeric(
    hosp_clean['overall_rating'], errors='coerce'
)

# standardize facility ID to 6-digit string with leading zeros
# e.g. 10001 becomes 010001
hosp_clean['facility_id'] = hosp_clean['facility_id'].astype(str).str.zfill(6)

print("\nHospital general info cleaned:")
print(hosp_clean.dtypes)
print("Missing ratings:", hosp_clean['overall_rating'].isnull().sum())


# ── 3. CLEAN READMISSION PROGRAM FILE ────────────────────────────────────────

# convert numeric columns - 'Too Few to Report' becomes NaN
readm_by_hospital['Excess Readmission Ratio'] = pd.to_numeric(
    readm_by_hospital['Excess Readmission Ratio'], errors='coerce'
)
readm_by_hospital['Predicted Readmission Rate'] = pd.to_numeric(
    readm_by_hospital['Predicted Readmission Rate'], errors='coerce'
)
readm_by_hospital['Expected Readmission Rate'] = pd.to_numeric(
    readm_by_hospital['Expected Readmission Rate'], errors='coerce'
)
readm_by_hospital['Number of Discharges'] = pd.to_numeric(
    readm_by_hospital['Number of Discharges'], errors='coerce'
)

# standardize facility ID to match hosp_clean
readm_by_hospital['Facility ID'] = readm_by_hospital['Facility ID'].astype(str).str.zfill(6)


# ── 4. COLLAPSE READMISSION FILE TO ONE ROW PER HOSPITAL ─────────────────────

# all six conditions - general hospital quality summary
all_conditions_summary = readm_by_hospital.groupby(['Facility ID', 'State']).agg(
    avg_excess_readm_ratio   = ('Excess Readmission Ratio',   'mean'),
    avg_predicted_readm_rate = ('Predicted Readmission Rate', 'mean'),
    avg_expected_readm_rate  = ('Expected Readmission Rate',  'mean'),
    num_conditions_measured  = ('Measure Name',               'count')
).reset_index().round(4)

# heart failure and COPD only - diabetes-relevant proxy
# diabetic patients have significantly higher comorbidity rates for both
diabetes_relevant = readm_by_hospital[
    readm_by_hospital['Measure Name'].isin(['READM-30-HF-HRRP', 'READM-30-COPD-HRRP'])
]

diabetic_proxy_summary = diabetes_relevant.groupby(['Facility ID', 'State']).agg(
    avg_excess_readm_hf_copd    = ('Excess Readmission Ratio',   'mean'),
    avg_predicted_readm_hf_copd = ('Predicted Readmission Rate', 'mean'),
    avg_expected_readm_hf_copd  = ('Expected Readmission Rate',  'mean')
).reset_index().round(4)

print("\nAll conditions summary shape:    ", all_conditions_summary.shape)
print("Diabetic proxy summary shape:    ", diabetic_proxy_summary.shape)


# ── 5. JOIN ALL THREE TABLES INTO ONE ENRICHED HOSPITAL TABLE ────────────────

# first join: hospital general info + all conditions summary
hosp_enriched = hosp_clean.merge(
    all_conditions_summary,
    left_on='facility_id',
    right_on='Facility ID',
    how='left'
).drop(columns=['Facility ID', 'State'])

hosp_enriched = hosp_enriched.rename(columns={'State_x': 'state'})

# second join: add the diabetes-relevant proxy columns
hosp_enriched = hosp_enriched.merge(
    diabetic_proxy_summary[['Facility ID',
                             'avg_excess_readm_hf_copd',
                             'avg_predicted_readm_hf_copd',
                             'avg_expected_readm_hf_copd']],
    left_on='facility_id',
    right_on='Facility ID',
    how='left'
).drop(columns=['Facility ID'])

print("\nFinal enriched hospital table shape:", hosp_enriched.shape)
print(hosp_enriched.head(3))


# ── 6. COMPUTE STATE-LEVEL AVERAGES ──────────────────────────────────────────

state_quality = hosp_enriched.groupby('state').agg(
    num_hospitals            = ('facility_id',              'count'),
    avg_star_rating          = ('overall_rating',           'mean'),
    avg_excess_readm_ratio   = ('avg_excess_readm_ratio',   'mean'),
    avg_excess_readm_hf_copd = ('avg_excess_readm_hf_copd', 'mean')
).reset_index().round(2)

print("\nState quality table shape:", state_quality.shape)
print(state_quality.sort_values('avg_star_rating', ascending=False).head(5))


# ── 7. COMPUTE NATIONAL BENCHMARKS ───────────────────────────────────────────

national_benchmark = pd.DataFrame([{
    'total_hospitals_rated':         int(hosp_enriched['overall_rating'].count()),
    'national_avg_star_rating':      round(hosp_enriched['overall_rating'].mean(), 2),
    'national_avg_excess_readm':     round(hosp_enriched['avg_excess_readm_ratio'].mean(), 4),
    'national_avg_predicted_readm':  round(hosp_enriched['avg_predicted_readm_rate'].mean(), 2),
    'national_avg_excess_hf_copd':   round(hosp_enriched['avg_excess_readm_hf_copd'].mean(), 4),
    'pct_hospitals_below_expected':  round(
        (hosp_enriched['avg_excess_readm_ratio'] < 1.0).sum() /
         hosp_enriched['avg_excess_readm_ratio'].count() * 100, 1
    )
}])

print("\nNational Benchmarks:")
print(national_benchmark.T)


# ── 8. SAVE ALL TABLES TO SQLITE ─────────────────────────────────────────────

os.makedirs('data', exist_ok=True)
connection = sqlite3.connect('data/diabetic_data.db')

df.to_sql(
    'diabetic_data', connection, if_exists='replace', index=False
)
hosp_enriched.to_sql(
    'hospital_quality', connection, if_exists='replace', index=False
)
state_quality.to_sql(
    'state_quality', connection, if_exists='replace', index=False
)
national_benchmark.to_sql(
    'national_benchmark', connection, if_exists='replace', index=False
)

connection.close()
print("\nAll four tables saved to data/diabetic_data.db")


# ── 9. VERIFY EVERYTHING LANDED CORRECTLY ────────────────────────────────────

connection = sqlite3.connect('data/diabetic_data.db')

audit = pd.read_sql("""
    SELECT 'diabetic_data'      AS table_name, COUNT(*) AS rows FROM diabetic_data
    UNION ALL
    SELECT 'hospital_quality'   AS table_name, COUNT(*) AS rows FROM hospital_quality
    UNION ALL
    SELECT 'state_quality'      AS table_name, COUNT(*) AS rows FROM state_quality
    UNION ALL
    SELECT 'national_benchmark' AS table_name, COUNT(*) AS rows FROM national_benchmark
""", connection)

connection.close()
print("\nDatabase audit:")
print(audit)